# 06 · GPU computing

*Notebooks: [01 Overview](01-overview.ipynb) · [02 Formats](02-formats.ipynb) · [03 Data](03-data.ipynb) · [04 Maps](04-maps.ipynb) · [05 Plots](05-plots.ipynb) · [06 Compute](06-compute.ipynb)*

`autk-compute` runs WebGPU computation over features. It has two engines, and pyautark builds both as `Compute`:

1. **`Compute.gpgpu`**: your WGSL function is run once per feature, with its properties as inputs, in parallel on the GPU.
2. **`Compute.render`**: renders the city from viewpoints and counts what each view sees (sky, buildings...).

In [ ]:
import autark as ak
import tutorial as t

## 1 A function per feature

The sample is 64 buildings of lower Manhattan with their `area` (m²), `perimeter` (m) and `floors`.

In [ ]:
buildings = ak.GeoJSON("buildings", url=t.url("data/compute/lower_mnt_buildings.geojson"))

import pandas as pd
frame = pd.DataFrame([f["properties"] for f in t.read_json("data/compute/lower_mnt_buildings.geojson")["features"]])
frame.head()

## 2 `Compute.gpgpu`

`Compute.gpgpu(source, variable_mapping=..., wgsl_body=..., result_field=...)`:

- `variable_mapping` maps **WGSL variable names** to **feature properties**; they become `f32` values in the shader;
- `wgsl_body` is the **body** of a function that returns the result (not a whole module; a string or a list of lines);
- `result_field` (one value) or `output_columns` (several) names where the result goes: `properties.compute.<name>`.

A floor-area proxy, `area * floors`, as the lightest example:

In [ ]:
floor_area = ak.Compute.gpgpu(
    buildings,
    variable_mapping={"area": "area", "floors": "floors"},
    wgsl_body="return area * floors;",
    result_field="floor_area",
)

ak.Spec(
    data=[buildings],
    transforms=[floor_area],
    views=[
        ak.Map(camera=ak.Camera(pitch=45), layers=[
            ak.Layer(buildings, id="buildings").encode(color=ak.field("compute.floor_area", ak.Scale(type="quantile", scheme="viridis")))
        ]),
        ak.Table(buildings, columns=["name", "area", "floors", "compute.floor_area"], sort={"column": "compute.floor_area", "direction": "desc"}),
    ],
    layout=ak.Layout(type="vertical"),
).widget()

Without `output`, the result is added to the source table. With `output="name"` it goes into a **new table** and the source
stays as it was. A join or heatmap cannot read a table a compute pass wrote.

### Scalar and multi-column outputs

A pass can return several values: `output_columns=["a", "b"]`, and the function returns an array with one `f32` per column: `array<f32, 2>(a, b)`.

In [ ]:
two = ak.Compute.gpgpu(
    buildings,
    output="metrics",
    variable_mapping={"area": "area", "perimeter": "perimeter"},
    wgsl_body="return array<f32, 2>(area / perimeter, perimeter * perimeter / area);",
    output_columns=["thickness", "shape_index"],
)
ak.Spec(
    data=[buildings],
    transforms=[two],
    views=[ak.Table("metrics", columns=["name", "compute.thickness", "compute.shape_index"])],
).widget()

### Example: building compactness

The isoperimetric quotient, 4πA / P², is 1 for a circle and shrinks for elongated shapes:

In [ ]:
compactness = ak.Compute.gpgpu(
    buildings,
    variable_mapping={"area": "area", "perimeter": "perimeter"},
    wgsl_body="return (4.0 * 3.1415927 * area) / (perimeter * perimeter);",
    result_field="compactness",
)

ak.Spec(
    data=[buildings],
    transforms=[compactness],
    views=[ak.Map(layers=[
        ak.Layer(buildings, id="compactness").encode(color=ak.field("compute.compactness", ak.Scale(scheme="spectral")))
    ])],
).widget()

## 3 Render analysis

`Compute.render` answers visibility questions. It draws the scene from each **viewpoint** in several directions on the GPU,
then counts the pixels of each class. The recipe:

- `layers`: what is in the scene (`source` table and `type`, such as `buildings` or `roads`);
- `viewpoints`: the table whose features are the viewpoints (`strategy` is the centroid unless you ask for
  `building-windows`), and `sampling` (`directions` around each viewpoint);
- `aggregation`: `classes` counts pixels per layer (with `includeBackground` for what no layer covers: the sky), `objects`
  counts per object;
- `output`: the new table, a copy of the viewpoints with the result in `compute.render`.

### Sky exposure per road

Place a viewpoint at the center of each road segment, render the buildings around it, and count the share of the view
that no building covers.

In [ ]:
osm = ak.OSM(
    "mnt", bbox=(-74.015, 40.703, -74.003, 40.713), layers=["roads", "buildings"],
    source="pbf", pbf_file_url=t.url("data/osm/lower_mnt.osm.pbf"),
)

sky = ak.Compute.render(
    output="sky",
    layers=[{"id": "buildings", "source": "mnt_buildings", "type": "buildings"}],
    viewpoints={"source": "mnt_roads", "sampling": {"directions": 8}},
    aggregation={"type": "classes", "includeBackground": True},
    tile_size=64,
)

ak.Spec(
    data=[osm],
    transforms=[sky],
    views=[ak.Map(camera=ak.Camera(zoom=13, pitch=45), layers=[
        ak.Layer("mnt_buildings", id="buildings").style(color="#bbbbbb", opacity=0.35),
        ak.Layer("sky", id="sky exposure").encode(
            color=ak.field("compute.render.classes.background", ak.Scale(scheme="blues"))
        ).style(width=14),
    ])],
).widget()

Narrow streets between tall buildings see little sky (dark); wide avenues and the waterfront see more (light). The
output table `sky` is a copy of the roads, so a layer, a plot or a table can read `compute.render.classes.background`
like any property.

## Where next

You have seen all four modules from Python. For the full use case, a city application with OpenStreetMap, several urban
datasets, GPU scores and linked views, see the *Urbane* example of the [Autark tutorial](https://github.com/ctsilva/autark-tutorial).